# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShardhaBatra/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

For my freestyle project, one row represents the daily performance of one content page for one client on one specific date from the `fact_content_daily_performance` table.

## Time Window

For feature development, I will use a mid-panel month (March 2026). This month is recommended because it is not part of the final outcome window. The final month (June 2026) will be reserved for testing and validation.

My project predicts future content decline using information that is available before the prediction date.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

These features are available before the prediction time and can help predict future content decline.

## Label
Future content decline (defined using a future time window).

The label represents the outcome that the model will predict.

## Context
- client_hash_id
- content_hash_id
- report_date

These fields are used for joining tables, grouping data, and validation. They are not used as model features because IDs do not contain predictive information.

## Excluded
- trend_direction
- trend_pct
- Any product decision flags (if available)

These fields are excluded because they directly reveal the target or are derived from it, which would cause data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
!pip install -q duckdb huggingface_hub pyarrow

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get("HF_TOKEN"))

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Total files:", len(files))

for f in files[:50]:
    print(f)

Total files: 24
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_

In [ ]:
from datasets import load_dataset
from google.colab import userdata

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=userdata.get("HF_TOKEN")
)

print(ds)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 9841378
})


In [ ]:
import pandas as pd

df = ds.to_pandas()

print(df.shape)

(9841378, 30)


In [ ]:
duplicates = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [ ]:
print("Total rows:", len(df))
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

Total rows: 9841378
Start date: 2026-03-01
End date: 2026-03-31


In [ ]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

missing = df[features].isnull().sum().to_frame("Missing Values")
missing["Percentage"] = (missing["Missing Values"] / len(df) * 100).round(2)

missing

,Missing Values,Percentage
gsc_impressions,0,0.00
gsc_clicks,0,0.00
gsc_avg_position,6230317,63.31
ga4_sessions,3018741,30.67
scroll_events,3018741,30.67


### Missing Values Observation

The selected features do not have the same level of completeness.

- `gsc_impressions` and `gsc_clicks` have no missing values.
- `gsc_avg_position` has about 63.31% missing values.
- `ga4_sessions` and `scroll_events` have about 30.67% missing values.

These missing values are expected because search and GA4 data are not available for every client or every day. Before building a model, I will investigate whether these missing values follow a pattern and handle them carefully instead of simply replacing them with zero.

## Feature Availability

### gsc_impressions
Knowable at the decision moment because historical impression data is already available before making a prediction.

### gsc_clicks
Knowable at the decision moment because past click data is collected before the prediction date.

### gsc_avg_position
Knowable at the decision moment because search ranking is observed before the future outcome occurs.

### ga4_sessions
Knowable at the decision moment because session counts come from historical user activity.

### scroll_events
Knowable at the decision moment because engagement events are recorded before the prediction window.

In [ ]:
available = df[df["ga4_data_available"] == True]

print("Rows with GA4 available:", len(available))

print("Percentage:",
      round(len(available)/len(df)*100,2), "%")

Rows with GA4 available: 413966
Percentage: 4.21 %


### Availability Observation

Only 4.21% of the rows have GA4 data available in this month.

This shows that not every client has GA4 data for every day. Therefore, I will always check the availability flag before using GA4-based features in my analysis.

## Leakage Demonstration
To understand data leakage, I intentionally add one label-derived feature (`trend_direction` or `trend_pct`) into a simple experiment. This should produce an unrealistically good result because the feature already contains information about the target. After observing this, I remove the feature and keep the honest version of the dataset for future modeling.

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/ShardhaBatra/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

starter = pd.read_csv(url)

starter[["trend_direction", "trend_pct"]].head()

,trend_direction,trend_pct
0,down,-41.4
1,down,-57.7
2,down,-60.9
3,stable,-13.8
4,down,-34.7


## Leakage Demonstration

For this demonstration, I intentionally inspect the label-derived columns `trend_direction` and `trend_pct`. These columns are created from the outcome itself, so using them as model features would leak future information into the model.

In [5]:
# Simulate adding a leakage feature

leak_demo = starter[[
    "trend_direction",
    "trend_pct"
]].copy()

print("Leakage features added:")
print(leak_demo.head())

Leakage features added:
  trend_direction  trend_pct
0            down      -41.4
1            down      -57.7
2            down      -60.9
3          stable      -13.8
4            down      -34.7


### Observation

The columns `trend_direction` and `trend_pct` are derived from the future trend of the content. If these columns were used as input features, the model would already know information about the target. This would produce unrealistically high performance and would not generalize to new data.

Therefore, these columns must be excluded from the final feature set.

In [6]:
# Remove leakage features

safe_features = starter.drop(columns=["trend_direction", "trend_pct"])

print("Leakage features removed.")
print("Remaining columns:", len(safe_features.columns))

Leakage features removed.
Remaining columns: 42


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset cannot prove that updating or refreshing a page will cause its traffic to recover. It only contains historical search and engagement signals, so the results can only support content review decisions. External factors such as Google algorithm updates, seasonality, competitor actions, or content quality changes are not fully captured in the data. Therefore, my work will provide decision-support recommendations, not causal proof.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.